# NeoOLAF native EventStoryLine layer ablation — one document v1.3

This v1.3 notebook keeps the same **OWL-Time** ontology used in RAGTree and the controlled labels `PRECONDITION`/`FALLING_ACTION`. Layer 1 now uses parallel sentence-level event inventories, two whole-document coverage reviews, and source-batched closed-inventory PLOT_LINK generation. Gold remains unavailable until after Layer 12.


In [ ]:
from __future__ import annotations

import os
import sys
from getpass import getpass
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").is_file() and (candidate / "src/neoolaf").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the NeoOLAF repository.")


def first_existing_path(env_name: str, candidates: list[Path]) -> Path:
    raw = os.environ.get(env_name, "").strip().strip('"').strip("'")
    if raw:
        path = Path(raw).expanduser().resolve()
        if path.is_file():
            return path
        raise FileNotFoundError(f"{env_name} points to a missing file: {path}")
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(
        f"Could not find {env_name}. Tried:\n"
        + "\n".join(str(Path(x).expanduser().resolve()) for x in candidates)
    )


PROJECT_ROOT = find_project_root()
NOTEBOOK_DIR = PROJECT_ROOT / "examples/RAGTreeDatasets"
TOOLS_DIR = NOTEBOOK_DIR / "tools"
for path in [PROJECT_ROOT / "src", PROJECT_ROOT, TOOLS_DIR]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from eventstoryline_native_ablation_v1_3 import (
    RELATION_IDS,
    analyze_run,
    gold_event_index,
    indexed_token_table,
    load_layer_states,
    project_event_label,
    read_json,
    read_jsonl,
    run_native_pipeline,
    seed_ontology_summary,
)

print("PROJECT_ROOT =", PROJECT_ROOT)

## 1. Configuration

In [ ]:
INPUT_JSONL = NOTEBOOK_DIR / "data/eventstoryline_one_input_v1.jsonl"
GOLD_JSONL = NOTEBOOK_DIR / "data/eventstoryline_one_gold_v1.jsonl"
SMOKE5_INPUT_JSONL = NOTEBOOK_DIR / "data/eventstoryline_smoke5_input_v1.jsonl"
SMOKE5_GOLD_JSONL = NOTEBOOK_DIR / "data/eventstoryline_smoke5_gold_v1.jsonl"

# Same seed ontology used by the RAGTree EventStoryLine experiments.
# Override explicitly with EVENTSTORYLINE_ONTOLOGY_PATH when needed.
ONTOLOGY_PATH = first_existing_path(
    "EVENTSTORYLINE_ONTOLOGY_PATH",
    [
        PROJECT_ROOT.parent / "ragtree/data/ontology/OWLTime/time.ttl",
        PROJECT_ROOT.parent / "RAGTree/data/ontology/OWLTime/time.ttl",
        PROJECT_ROOT / "../ragtree/data/ontology/OWLTime/time.ttl",
        PROJECT_ROOT / "../RAGTree/data/ontology/OWLTime/time.ttl",
    ],
)

# Controlled normalized benchmark relation schema. This is not the seed ontology.
RELATION_CATALOG = NOTEBOOK_DIR / "ontology/eventstoryline_relation_catalog.json"
RELATION_ALIASES = NOTEBOOK_DIR / "ontology/eventstoryline_relation_aliases.json"
PROFILE_PATH = NOTEBOOK_DIR / "configs/eventstoryline_profile_native_ablation_v1_3.json"
GUIDANCE_PATH = NOTEBOOK_DIR / "configs/guidance_eventstoryline_native_ablation_v1_3.json"
TASK_GUIDANCE_PATH = NOTEBOOK_DIR / "configs/eventstoryline_task_guidance_v1_3.json"

RUNS_ROOT = NOTEBOOK_DIR / "runs/eventstoryline_native_layer_ablation"
RUN_DIR = RUNS_ROOT / "document_1_10ecbplus_v1_3_owltime_recall"

OPENROUTER_HOST = "https://openrouter.ai/api/v1"
MODEL_NAME = "openai/gpt-oss-20b"
API_KEY = os.environ.get("OPENROUTER_API_KEY", "").strip().strip('"').strip("'")

# Relation-instance decisions are independent and run concurrently in Layer 2.
WORKERS = 16
REASONING_EFFORT = "minimal"
RUN_PIPELINE = True
CLEAN_RUN_DIR = True

print("Input:", INPUT_JSONL)
print("Gold:", GOLD_JSONL)
print("OWL-Time seed ontology:", ONTOLOGY_PATH)
print("Controlled relation catalog:", RELATION_CATALOG)
print("Profile:", PROFILE_PATH)
print("Guidance:", GUIDANCE_PATH)
print("Task guidance:", TASK_GUIDANCE_PATH)
print("Run dir:", RUN_DIR)
print("Model:", MODEL_NAME)
print("API key available:", bool(API_KEY))


## 2. Preflight: anti-leakage, OWL-Time seed ontology, five-document files and event identity


In [ ]:
required = [
    INPUT_JSONL, GOLD_JSONL, SMOKE5_INPUT_JSONL, SMOKE5_GOLD_JSONL,
    ONTOLOGY_PATH, RELATION_CATALOG, RELATION_ALIASES,
    PROFILE_PATH, GUIDANCE_PATH, TASK_GUIDANCE_PATH,
]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError("Missing required files:\n" + "\n".join(missing))

input_rows = read_jsonl(INPUT_JSONL)
gold_rows = read_jsonl(GOLD_JSONL)
smoke_input_rows = read_jsonl(SMOKE5_INPUT_JSONL)
smoke_gold_rows = read_jsonl(SMOKE5_GOLD_JSONL)
assert len(input_rows) == 1 and len(gold_rows) == 1
assert len(smoke_input_rows) == 5 and len(smoke_gold_rows) == 5
assert "entities" not in input_rows[0] and "relations" not in input_rows[0]
assert all("entities" not in row and "relations" not in row for row in smoke_input_rows)
assert input_rows[0]["document_id"] == gold_rows[0]["document_id"]
assert [x["document_id"] for x in smoke_input_rows] == [x["document_id"] for x in smoke_gold_rows]

profile = read_json(PROFILE_PATH)
task = read_json(TASK_GUIDANCE_PATH)
catalog = read_json(RELATION_CATALOG)
gold = gold_rows[0]
seed_summary = seed_ontology_summary(ONTOLOGY_PATH)

print("Document:", input_rows[0]["document_id"], "-", input_rows[0]["title"])
print("Source characters:", len(input_rows[0]["text"]))
print("Sentences:", len(input_rows[0]["sentences"]))
print("Source tokens:", sum(len(x) for x in input_rows[0]["tokens"]))
print("Gold events (not exposed to pipeline):", len(gold["entities"]))
print("Gold evaluated relations:", sum(len(v) for k, v in gold["relations"].items() if k in RELATION_IDS))
print("Ignored null pairs:", len(gold["relations"].get("null", [])))
print("OWL-Time classes loaded:", seed_summary["class_count"])
print("OWL-Time properties loaded:", seed_summary["property_count"])
print("Controlled task relations:", catalog["property_count"])
print("Relation IDs:", task["allowed_relation_ids"])
print("Layer 1 sentence workers:", profile["layers"]["layer01_linguistic_expression_extraction"]["sentence_workers"])
print("Layer 1 coverage reviews:", profile["layers"]["layer01_linguistic_expression_extraction"]["coverage_review_passes"])
print("Layer 1 relation workers:", profile["layers"]["layer01_linguistic_expression_extraction"]["relation_workers"])
print("Layer 1 source batch size:", profile["layers"]["layer01_linguistic_expression_extraction"]["relation_source_batch_size"])
print("Layer 2 workers:", profile["layers"]["layer02_candidate_enrichment"]["max_concurrency"])
print("Mention-free relation schemas injected:", len(profile["relations"]["allowed"]))
print("Five-document input IDs:", [row["document_id"] for row in smoke_input_rows])

assert seed_summary["class_count"] > 0 or seed_summary["property_count"] > 0
assert catalog["property_count"] == 2
assert set(task["allowed_relation_ids"]) == set(RELATION_IDS)
assert profile["relations"]["allowed"] == []
assert profile["anti_cheating"]["direct_eventstoryline_extraction"] is False
assert profile["anti_cheating"]["source_event_anchoring"] is False
assert profile["anti_cheating"]["gold_pair_hints"] is False
assert profile["anti_cheating"]["post_run_relation_invention"] is False
assert profile["benchmark_projection"]["gold_available_to_pipeline"] is False
assert profile["benchmark_projection"]["same_seed_ontology_as_ragtree"] is True

display(Markdown("### Indexed source table used by Layer 1"))
print(indexed_token_table(input_rows[0]["sentences"], input_rows[0]["tokens"]))

assert profile["anti_cheating"]["gold_event_lexicon"] is False
assert profile["anti_cheating"]["gold_event_count"] is False
assert profile["anti_cheating"]["gold_relation_count"] is False


## 3. Run the full native Layer 0--12 pipeline

In [ ]:
if RUN_PIPELINE:
    if not API_KEY:
        API_KEY = getpass("OpenRouter API key: ").strip().strip('"').strip("'")
    if not API_KEY:
        raise RuntimeError("No OpenRouter API key was provided.")

    final_state = run_native_pipeline(
        project_root=PROJECT_ROOT,
        input_jsonl=INPUT_JSONL,
        ontology_path=ONTOLOGY_PATH,
        profile_path=PROFILE_PATH,
        guidance_path=GUIDANCE_PATH,
        task_guidance_path=TASK_GUIDANCE_PATH,
        relation_catalog_path=RELATION_CATALOG,
        relation_aliases_path=RELATION_ALIASES,
        run_dir=RUN_DIR,
        model_name=MODEL_NAME,
        api_key=API_KEY,
        host=OPENROUTER_HOST,
        workers=WORKERS,
        reasoning_effort=REASONING_EFFORT,
        verbose=True,
        clean_run_dir=CLEAN_RUN_DIR,
    )
    print("Full native run completed.")
else:
    print("RUN_PIPELINE=False: reusing", RUN_DIR)

### Runtime evidence saved

- `run_manifest.json` records OWL-Time and the v1.3 operational fingerprint.
- `run_logs/layer01_call_audit.json` records sentence calls, coverage reviews, and source batches.
- `run_logs/layer01_event_inventory.json` records every accepted/rejected span proposal.
- `run_logs/layer01_relation_generation.json` records closed-inventory pair proposals.
- `run_logs/layer01_event_relation_instances.json` records materialized expressions.
- Layer 2/4 decisions, ontology retrieval, API responses, and all Layer 0--12 states remain saved.


## 4. Strict event and relation evaluation

In [ ]:
summary = analyze_run(
    run_dir=RUN_DIR,
    gold_jsonl=GOLD_JSONL,
    catalog_path=RELATION_CATALOG,
    aliases_path=RELATION_ALIASES,
)

display(pd.DataFrame(summary["layer_summary"]))

print("Strict relation evaluation; null relations excluded")
display(pd.DataFrame([summary["strict_relation_evaluation"]]))

print("Event mention inventory evaluation")
display(pd.DataFrame([summary["event_entity_evaluation"]]))

print("Relation-endpoint event inventory evaluation")
display(pd.DataFrame([summary["relation_endpoint_evaluation"]]))

print("Per-relation metrics")
display(pd.DataFrame(summary["per_relation_metrics"]))

print("Cumulative evaluation")
display(pd.DataFrame(summary["cumulative_evaluation"]))

print("First-failure counts")
print(summary["failure_counts"])

## 5. Layer 1 validated event inventory and closed-inventory relation instances

In [ ]:
layer1_rows = read_json(RUN_DIR / "run_logs/layer01_event_relation_instances.json")
layer1_df = pd.DataFrame(layer1_rows)
display(layer1_df)

accepted = layer1_df[layer1_df["status"] == "accepted"] if not layer1_df.empty else layer1_df
if not accepted.empty:
    print("Accepted event mentions:", int((accepted["label"] == "event_mention").sum()))
    print("Accepted relation instances:", int((accepted["label"] == "relation_instance").sum()))
    print("Rejected rows:", int((layer1_df["status"] == "rejected").sum()))

### Layer 1A span validation/repair and Layer 1B closed-inventory audit


In [ ]:
inventory_audit = pd.DataFrame(read_json(RUN_DIR / "run_logs/layer01_event_inventory.json"))
relation_generation = pd.DataFrame(read_json(RUN_DIR / "run_logs/layer01_relation_generation.json"))
call_audit = pd.DataFrame(read_json(RUN_DIR / "run_logs/layer01_call_audit.json"))

print("Layer 1A event inventory audit")
display(inventory_audit)
if not inventory_audit.empty:
    print("Accepted validated events:", int((inventory_audit["status"] == "accepted").sum()))
    print("Rejected event proposals:", int((inventory_audit["status"] == "rejected").sum()))
    if "repaired" in inventory_audit:
        print("Source-token span repairs:", int(inventory_audit["repaired"].fillna(False).sum()))

print("Layer 1B relation generation audit")
display(relation_generation)
if not relation_generation.empty:
    print("Accepted closed-inventory pairs:", int((relation_generation["status"] == "accepted").sum()))
    print("Rejected pair proposals:", int((relation_generation["status"] == "rejected").sum()))

print("Layer 1 sentence/review/source-batch call audit")
display(call_audit)
if not call_audit.empty:
    display(call_audit.groupby(["phase", "status"], dropna=False).size().reset_index(name="calls"))


## 6. Parallel Layer 2 ontology decisions

In [ ]:
layer2 = pd.DataFrame(read_json(RUN_DIR / "run_logs/layer02_relation_decisions.json"))
prompt_audit = pd.DataFrame(read_json(RUN_DIR / "run_logs/layer02_compact_prompt_audit.json"))

display(layer2)
print("Compact prompt audit")
display(prompt_audit)
if not prompt_audit.empty:
    print("Layer 2 calls:", len(prompt_audit))
    print("Mean Layer 2 user characters:", round(prompt_audit["user_chars"].mean(), 1))
    print("Maximum Layer 2 user characters:", int(prompt_audit["user_chars"].max()))
    print("Candidates per decision:", sorted(set(len(x) for x in prompt_audit["candidate_relation_ids"])))

## 7. Strict gold-relation trace

In [ ]:
trace_df = pd.read_csv(RUN_DIR / "analysis/gold_relation_trace.csv")
display(trace_df)
display(
    trace_df.groupby("first_failure", dropna=False)
    .size()
    .reset_index(name="gold_relations")
    .sort_values("gold_relations", ascending=False)
)

## 8. Native event candidates, assertions and triples

In [ ]:
states = {index: state for index, _, state in load_layer_states(RUN_DIR)}
layer3 = states.get(3)
layer4 = states.get(4)
layer5 = states.get(5)
layer6 = states.get(6)
layer11 = states.get(11)

if layer3:
    print("Layer 3 event candidates")
    display(pd.DataFrame([{
        "candidate_id": c.candidate_id,
        "canonical_label": c.canonical_label,
        "mentions": [m.text for m in c.mentions],
        "ontology_hints": c.ontology_hints,
    } for c in layer3.event_candidates or []]))

    print("Layer 3 relation candidates")
    display(pd.DataFrame([{
        "candidate_id": c.candidate_id,
        "canonical_label": c.canonical_label,
        "mentions": [m.text for m in c.mentions],
        "controlled_hints": [h for h in c.ontology_hints if str(h).lower().startswith("controlled_relation:")],
    } for c in layer3.relation_candidates or []]))
    assert all(c.mentions for c in layer3.relation_candidates or []), "Mention-free relation candidate detected."

if layer4:
    print("Layer 4 assertions")
    display(pd.DataFrame([{
        "source": x.source_candidate_label,
        "predicate": x.relation_label,
        "target": x.target_candidate_label,
        "confidence": x.confidence,
    } for x in layer4.candidate_relation_assertions or []]))

if layer5:
    print("Layer 5 triples")
    display(pd.DataFrame([{
        "subject": x.subject_label,
        "predicate": x.predicate_label,
        "object": x.object_label,
        "confidence": x.confidence,
    } for x in layer5.candidate_triples or []]))

print("Layer 6 ontology relation candidates:", len(layer6.ontology_relation_candidates or []) if layer6 else None)
print("Layer 11 completion candidates:", len(layer11.completion_candidates or []) if layer11 else None)

## 9. Event projection audit

In [ ]:
projection = pd.read_csv(RUN_DIR / "analysis/event_projection_audit.csv")
display(projection)

# Exact-span demonstration using source structure. Gold is consulted only here,
# after the pipeline has completed.
first_gold_key = next(iter(gold_event_index(gold)["keys_by_id"].values()))[0]
print(first_gold_key)
print(project_event_label(first_gold_key, gold))

## 10. Speed, concurrency and errors

In [ ]:
def optional_jsonl(path: Path) -> pd.DataFrame:
    return pd.DataFrame(read_jsonl(path)) if path.is_file() else pd.DataFrame()

calls = optional_jsonl(RUN_DIR / "run_logs/llm_calls.jsonl")
errors = optional_jsonl(RUN_DIR / "run_logs/llm_errors.jsonl")
parse_errors = optional_jsonl(RUN_DIR / "run_logs/llm_parse_errors.jsonl")
retrieval = optional_jsonl(RUN_DIR / "run_logs/ontology_retrieval.jsonl")

if not calls.empty:
    display(calls)
    display(calls.groupby("layer_tag").agg(
        calls=("call_index", "count"),
        total_recorded_seconds=("elapsed_seconds", "sum"),
        maximum_call_seconds=("elapsed_seconds", "max"),
        mean_system_chars=("system_chars", "mean"),
        mean_user_chars=("user_chars", "mean"),
        mean_response_chars=("response_chars", "mean"),
    ).reset_index())

print("Backend/API errors:", len(errors))
if not errors.empty: display(errors)
print("JSON parse errors:", len(parse_errors))
if not parse_errors.empty: display(parse_errors)
if not retrieval.empty:
    display(retrieval.groupby("layer_name").size().reset_index(name="retrieval_calls"))

manifest = read_json(RUN_DIR / "run_manifest.json")
print("Wall-clock pipeline seconds:", manifest.get("elapsed_seconds"))

## Success checklist before the five-document batch

1. The sibling RAGTree OWL-Time file resolves.
2. Parallel sentence extraction plus two whole-document reviews materially improves exact event recall.
3. Repeated mentions, eventive nominals, states, phrasal verbs, and hyphenated triggers retain exact indexed spans.
4. Layer 1B covers all validated sources in bounded batches and creates only closed-inventory pairs.
5. Layer 2 selects only `PRECONDITION`, `FALLING_ACTION`, or `found=false`; Layer 4 preserves direction.
6. Gold remains unavailable until post-Layer-12 evaluation. Freeze v1.3 before smoke-5 tuning.
